# Set Operations: UNION, INTERSECT, EXCEPT

## Overview
Set operations combine the results of two or more SELECT statements vertically (stacking rows), unlike JOINs which combine horizontally (adding columns).

**Rules for all set operations:**
- Both SELECT statements must return the **same number of columns**
- Corresponding columns must have **compatible data types**
- Column names in the result come from the **first** SELECT
- ORDER BY can only appear **once**, at the very end

**The four set operations:**

| Operation | Returns | Duplicates |
|---|---|---|
| `UNION` | All rows from both queries | Removes duplicates (like DISTINCT) |
| `UNION ALL` | All rows from both queries | Keeps all rows including duplicates |
| `INTERSECT` | Only rows present in both queries | Removes duplicates |
| `EXCEPT` | Rows in first query not in second | Removes duplicates |

**Performance note:** `UNION` is slower than `UNION ALL` because it must sort and deduplicate. Use `UNION ALL` unless you specifically need deduplication — and prefer `UNION ALL` + `GROUP BY` for aggregation after combining.

**SQLite support:** All four operations are supported.  
**PostgreSQL:** All four operations plus `INTERSECT ALL` and `EXCEPT ALL` (keep duplicates).

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    dob TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT,
    province TEXT, supervisor_id INTEGER
);
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_provider_id INTEGER
);
CREATE TABLE encounters (
    enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER,
    dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE referrals (
    referral_id INTEGER PRIMARY KEY, from_provider INTEGER, to_provider INTEGER,
    patient_id INTEGER, referral_date TEXT, reason TEXT
);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),(2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),(4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),(6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),(8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),(10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),
  (11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON',10),
  (13,'Dr. Ethan Tremblay','Orthopaedics','QC',10),
  (14,'Dr. Priya Nair','Emergency Medicine','BC',11),
  (15,'Dr. Marcus Okafor','Radiology','ON',11),
  (16,'Dr. Unassigned','Neurology','NB',12);
INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),(2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),(4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),(6,'Radiology','Building B',15),(7,'Neurology','Building C',16);
INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),
  ('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10','Essential hypertension','Cardiovascular'),
  ('R55','Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');
INSERT INTO encounters VALUES
  (1,1,10,1,'2023-01-15','J18.9',1850.00,1),(2,2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3,3,12,3,'2023-03-05','Z00.0',120.00,0),(4,4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5,5,14,5,'2023-04-02','J06.9',95.00,0),(6,6,10,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,1,'2023-06-22','I10',2100.00,1),(8,8,12,3,'2023-07-14',NULL,80.00,0),
  (9,1,14,5,'2023-08-30','R55',410.00,0),(10,3,12,3,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),(12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,90.00,0),(14,5,13,4,'2023-12-05','M54.5',450.00,0);
INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),
  (2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),
  (4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()

# Add a second encounter source to make UNION demos meaningful
conn.executescript("""
CREATE TABLE urgent_referrals (
    ref_id      INTEGER PRIMARY KEY,
    patient_id  INTEGER,
    provider_id INTEGER,
    ref_date    TEXT,
    reason      TEXT,
    cost_cad    REAL
);
INSERT INTO urgent_referrals VALUES
  (101, 1, 10, '2023-09-01', 'Chest pain escalation', 850.00),
  (102, 6, 14, '2023-05-15', 'Syncope workup', 310.00),
  (103, 3, 10, '2023-03-12', 'Cardiology consult', 650.00),
  (104, 9, 14, '2023-10-05', 'Emergency follow-up', 200.00);
""")
conn.commit()
print("Schema ready — including urgent_referrals table for UNION demos")


---
## UNION and UNION ALL

In [ ]:
# Combine regular encounters and urgent referrals into one event list
print("=== All patient events: encounters UNION ALL urgent referrals ===")
q = """
SELECT  enc_id          AS event_id,
        patient_id,
        enc_date        AS event_date,
        cost_cad,
        'encounter'     AS event_type
FROM    encounters

UNION ALL

SELECT  ref_id          AS event_id,
        patient_id,
        ref_date        AS event_date,
        cost_cad,
        'urgent_referral' AS event_type
FROM    urgent_referrals

ORDER BY event_date, patient_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

# UNION (vs UNION ALL): removes duplicates across the combined result
print()
print("=== Patient IDs who appear in EITHER table (UNION — deduplicates) ===")
q2 = """
SELECT patient_id FROM encounters
UNION
SELECT patient_id FROM urgent_referrals
ORDER BY patient_id
"""
q3 = """
SELECT patient_id FROM encounters
UNION ALL
SELECT patient_id FROM urgent_referrals
ORDER BY patient_id
"""
r2 = pd.read_sql(q2, conn)
r3 = pd.read_sql(q3, conn)
print(f"UNION  (distinct patient_ids): {len(r2)} rows")
print(f"UNION ALL (all rows):          {len(r3)} rows")
print(r2.to_string(index=False))


---
## INTERSECT: rows present in both queries

In [ ]:
# Patients who appear in BOTH encounters AND urgent_referrals
print("=== Patients with both an encounter AND an urgent referral (INTERSECT) ===")
q = """
SELECT patient_id FROM encounters
INTERSECT
SELECT patient_id FROM urgent_referrals
ORDER BY patient_id
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))

# Enrich with names
print()
print("=== Names of those patients ===")
ids = tuple(result['patient_id'].tolist())
q2 = f"""
SELECT patient_id, first_name || ' ' || last_name AS name, province
FROM   patients
WHERE  patient_id IN {ids}
ORDER BY patient_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# PostgreSQL INTERSECT ALL (keeps duplicates — rarely used)
print()
print("=== PostgreSQL INTERSECT ALL (reference) ===")
print("""
-- INTERSECT ALL keeps multiple occurrences:
-- If patient 1 appears 3 times in A and 2 times in B,
-- INTERSECT ALL returns 2 rows for patient 1.
-- Rarely needed in practice; INTERSECT (distinct) is almost always the right choice.
""")


---
## EXCEPT: rows in first query but not second

In [ ]:
# Patients who have encounters but NO urgent referrals
print("=== Patients with encounters but NO urgent referral (EXCEPT) ===")
q = """
SELECT patient_id FROM encounters
EXCEPT
SELECT patient_id FROM urgent_referrals
ORDER BY patient_id
"""
result = pd.read_sql(q, conn)
print(f"Patient IDs: {result['patient_id'].tolist()}")
print(result.to_string(index=False))

# Patients with urgent referrals but no regular encounters
print()
print("=== Patients with urgent referrals but NO regular encounter (EXCEPT reversed) ===")
q2 = """
SELECT patient_id FROM urgent_referrals
EXCEPT
SELECT patient_id FROM encounters
ORDER BY patient_id
"""
result2 = pd.read_sql(q2, conn)
print(result2.to_string(index=False))
if len(result2) == 0:
    print("All urgent referral patients also have regular encounters in this dataset.")

# Comparison: EXCEPT vs anti-join (LEFT JOIN WHERE IS NULL)
print()
print("=== EXCEPT vs anti-join — equivalent results ===")
q_except = """
SELECT patient_id FROM encounters
EXCEPT
SELECT patient_id FROM urgent_referrals
"""
q_antijoin = """
SELECT DISTINCT e.patient_id
FROM   encounters AS e
LEFT JOIN urgent_referrals AS ur ON e.patient_id = ur.patient_id
WHERE  ur.patient_id IS NULL
"""
r_e = set(pd.read_sql(q_except,   conn)['patient_id'].tolist())
r_a = set(pd.read_sql(q_antijoin, conn)['patient_id'].tolist())
print(f"EXCEPT result:    {sorted(r_e)}")
print(f"Anti-join result: {sorted(r_a)}")
print(f"Results match: {r_e == r_a}")


---
## Column alignment and common use patterns

In [ ]:
# Real-world use: combine data from multiple source systems
# Healthcare: events from two different hospital information systems
print("=== Unified patient timeline from two systems ===")
q = """
SELECT  'HIS_A'          AS source_system,
        enc_id           AS event_id,
        patient_id,
        enc_date         AS event_date,
        COALESCE(diag_code, 'UNCODED')  AS code,
        cost_cad,
        CASE admitted WHEN 1 THEN 'Inpatient' ELSE 'Outpatient' END AS visit_type
FROM    encounters

UNION ALL

SELECT  'HIS_B'          AS source_system,
        ref_id           AS event_id,
        patient_id,
        ref_date         AS event_date,
        'REFERRAL'       AS code,
        cost_cad,
        'Referral'       AS visit_type
FROM    urgent_referrals

ORDER BY patient_id, event_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. UNION removes duplicates and is slower than UNION ALL**
`UNION` implicitly applies DISTINCT across the combined result — it sorts or hashes all rows to find duplicates. Unless you specifically need deduplication, always use `UNION ALL`. It's faster and the intent is explicit.

**2. Column count mismatch raises an error with no hint about which column**
If the first SELECT has 5 columns and the second has 4, the error is simply `SELECTs to the left and right of UNION do not have the same number of result columns`. Count columns carefully. Use `NULL AS col_name` as a placeholder in the query that lacks a column.

**3. Column names come from the first SELECT — the second is silently ignored**
`SELECT patient_id, enc_date FROM encounters UNION SELECT patient_id, ref_date FROM urgent_referrals` — the result column is named `enc_date`, not `ref_date`. This is legal SQL but confusing. Use a neutral alias in the first SELECT: `SELECT patient_id, enc_date AS event_date ...`.

**4. ORDER BY at the end applies to the combined result, not each branch**
You cannot write `ORDER BY` inside each SELECT branch of a UNION. The ORDER BY must come after the final SELECT, and it applies to the whole combined result. To sort differently within each branch before combining, use a CTE or subquery per branch.

**5. EXCEPT is not commutative — order matters**
`A EXCEPT B` and `B EXCEPT A` return different results. `A EXCEPT B` gives rows in A not in B; `B EXCEPT A` gives rows in B not in A. This is a frequent source of logic errors when refactoring queries.

**6. INTERSECT/EXCEPT can be slow on large tables without indexes**
Both operations typically require sorting or hashing both result sets to identify common or missing rows. On large tables, an equivalent `EXISTS` subquery or indexed join often performs better. Always check `EXPLAIN ANALYZE` if performance is a concern.


---
*sql_methods_library - Samantha McGarrigle*